# STRUCTURED EXTRACTION

**Transforming Unstructured Invoice Images into Structured JSON Using OCR and Rule-Based Information Extraction.**

This notebook walks through the whole workflow step by step, using a few real invoice images from the FATURA dataset:

```
Invoice Images -> Image Preprocessing -> OCR -> Rule-Based Information Extraction -> Structured JSON
```

Before running: make sure `DATASET_DIR` and `TESSERACT_CMD` in `config.py` are correct for your computer, then choose **Run All**.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

# The notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from PIL import Image

import config
from src import image_processing as ip
from src import ocr_engine as ocr
from src import information_extraction as ie
from src import json_generator as jg

try:
    from IPython.display import display
except ImportError:          # lets the cells also run outside Jupyter
    display = print


def show(array, title, width=500):
    """Display a numpy image inside the notebook."""
    print(title)
    pil = Image.fromarray(array)
    pil.thumbnail((width, 10_000))
    display(pil)


tesseract_path = ocr.configure_tesseract(config.TESSERACT_CMD)
print("Tesseract:", tesseract_path, "| version", ocr.get_tesseract_version())
print("Images folder:", config.IMAGES_DIR)

## 1. Image loading
We pick the configured number of invoice images (JPG, JPEG or PNG) from the FATURA `images/` folder. The annotation files are **not** used.

In [ ]:
image_paths = ip.find_images(config.IMAGES_DIR, config.NUM_IMAGES,
                             config.SUPPORTED_EXTENSIONS, config.SELECTION_MODE)
print(f"{len(image_paths)} invoice images selected:")
for p in image_paths:
    print("  -", p.name)

## 2. Image preprocessing
Grayscale -> resize -> noise removal -> thresholding. Shown here for the first image.

In [ ]:
first_path = image_paths[0]
first_img = ip.load_image(first_path)
assert first_img is not None, f"Could not read {first_path}"

steps = ip.preprocess_steps(first_img, config.RESIZE_TARGET_WIDTH,
                            config.DENOISE_KERNEL, config.THRESHOLD_METHOD)
show(first_img[:, :, ::-1], f"Original: {first_path.name}")      # BGR -> RGB for display
show(steps["gray"], "1) Grayscale")
show(steps["resized"], f"2) Resized to width {config.RESIZE_TARGET_WIDTH}")
show(steps["denoised"], "3) Noise removal (median blur)")
show(steps["binary"], f"4) Thresholding ({config.THRESHOLD_METHOD})")

## 3. OCR with Tesseract
The thresholded image is converted to text. This is the *raw, unstructured* text.

In [ ]:
raw_text = ocr.extract_text(steps["binary"], config.TESSERACT_LANG, config.TESSERACT_CONFIG)
print(raw_text)

## 4. Rule-based information extraction
Regular expressions and keyword rules turn the raw text into fields. Missing text stays `""` and missing numbers stay `null` - nothing is invented.

In [ ]:
fields = ie.extract_invoice_fields(raw_text)
record = jg.build_invoice_record(first_path.stem, fields, raw_text)
print("Valid structure:", jg.validate_record(record) == [])
print(json.dumps({k: v for k, v in record.items() if k != "ocr_text"}, indent=2, ensure_ascii=False))

## 5. Run the full pipeline on every selected image
For each invoice we save the raw OCR text (`output/raw_text/`), one JSON file (`output/json/`), and finally one combined file (`output/all_invoices.json`). Unreadable images are skipped.

In [ ]:
records, skipped = [], []
for path in image_paths:
    img = ip.load_image(path)
    if img is None:
        skipped.append(path.name)
        print("SKIPPED (unreadable):", path.name)
        continue
    binary = ip.preprocess_image(img, config.RESIZE_TARGET_WIDTH, config.DENOISE_KERNEL, config.THRESHOLD_METHOD)
    text = ocr.extract_text(binary, config.TESSERACT_LANG, config.TESSERACT_CONFIG)
    jg.save_raw_text(path.stem, text, config.RAW_TEXT_DIR)
    rec = jg.build_invoice_record(path.stem, ie.extract_invoice_fields(text), text)
    assert jg.validate_record(rec) == [], path.name
    jg.save_invoice_json(rec, config.JSON_DIR)
    records.append(rec)

jg.save_all_invoices(records, config.ALL_INVOICES_FILE)
print(f"\nProcessed {len(records)} invoice(s); skipped {len(skipped)}.")
print("Combined file:", config.ALL_INVOICES_FILE)

## 6. Results overview

In [ ]:
print(f"{'invoice_id':32} {'number':14} {'date':14} {'items':>5} {'total':>12}  currency")
print("-" * 92)
for r in records:
    fin = r["financial"]
    total = "" if fin["total"] is None else f"{fin['total']:.2f}"
    print(f"{r['invoice_id'][:32]:32} {r['invoice_number'][:14]:14} {r['invoice_date'][:14]:14} "
          f"{len(r['items']):>5} {total:>12}  {fin['currency']}")

### Summary

Messy invoice images were converted into structured JSON data:

**Invoice Images -> Preprocessing -> OCR -> Rule-Based Extraction -> Structured JSON**

Rule-based extraction is simple and transparent, but it will not be perfect on every FATURA template (see *Limitations* in `README.md`).